> **교육생용 정답 코드**
> Python 3 커널을 선택하고 위에서부터 셀을 실행하세요.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False
plt.rc("font", family="Malgun Gothic")


# [Project] 공항기상관측(AMOS), 종관기상관측(ASOS) 데이터 교차 분석

---

## 프로젝트 목표

- 인천과 울산의 종관기상관측(ASOS) 데이터를 분석하고, 인천공항, 울산공항의 공항기상관측(AMOS) 데이터와 비교함으로써 기초적인 데이터 분석 기법을 연습합니다.

---

## 프로젝트 목차
1. **데이터 읽기:** 공항기상관측(AMOS), 종관기상관측(ASOS) 기상청 데이터를 불러오고 DataFrame 구조를 확인<br>
    1.1. 데이터 불러오기<br>
    1.2. 데이터 확인하기<br>
<br>
2. **데이터 정제:** 데이터 확인 후 측정 지점별 데이터 분리 및 결측치 제거<br>
    2.1. 데이터 분리<br>
    2.2. 결측치 제거<br>
<br>
3. **종관기상관측(ASOS) 데이터 분석:** 종관기상관측(ASOS) 데이터를 분석<br>
    3.1. 평균기온 비교<br>
    3.2. 기온과 해면기압의 상관관계 분석<br>
<br>
4. **AMOS 데이터, ASOS 데이터 교차분석:** 공항기상관측(AMOS) 데이터와 종관기상관측(ASOS) 데이터 비교<br>
    4.1. AMOS 데이터 불러오기<br>
    4.2. 평균기온 비교<br>
    4.3. 해면기압 데이터 비교<br>

---

## 데이터 출처
- 출처: 기상청 기상자료개방포털
- 수집 기간: 2024년 1월 1일 ~ 2024년 12월 31일
- 수집 대상:
  - AMOS: 최고기온(°C), 최저기온(°C), 평균해면기압(hPa), 최고해면기압(hPa), 최저해면기압(hPa)
  - ASOS: 평균기온(°C), 최저기온(°C), 최고기온(°C), 일강수량(mm), 최고 해면기압(hPa), 최저 해면기압(hPa), 평균 해면기압(hPa)

---

## 1. 데이터 읽기

필요한 패키지를 import 한 후 `pandas`를 사용하여 데이터를 읽고 어떠한 데이터가 저장되어 있는지 확인합니다.

### 1.1. 데이터 불러오기

In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

먼저, 종관기상관측(ASOS) 기상청 데이터를 불러와 `asos_all`에 저장합니다.

In [ ]:
# pd.read_csv를 통하여 종관기상관측(ASOS) 기상청 데이터를 데이터프레임 형태로 읽어옵니다.
asos_all = pd.read_csv("dataset/asos_rawdata.csv")

In [ ]:
# 종관기상관측(ASOS) 상위 5개 데이터를 출력합니다.
asos_all.head()

측정 시점을 나타내는 `일시` 칼럼은 `datetime` 형식으로 변환하고, 데이터의 구조를 확인합니다.

In [ ]:
asos_all["일시"] = pd.to_datetime(asos_all["일시"])

# 데이터프레임 정보를 요약하여 출력합니다. 
asos_all.info()

다른 데이터는 결측치가 없는 것으로 보이나, `일강수량(mm)` 데이터는 결측치가 많은 것 같습니다. 추후 결측치를 처리하도록 하겠습니다.

### 1.2. 데이터 확인하기

불러온 데이터프레임의 특정 칼럼 데이터를 **중복 없이 오름차순 정렬하여** 확인해 봅시다.

In [ ]:
unique_time_list = sorted(list(set(asos_all['일시']))) # 측정 일시 종류 확인
print(unique_time_list) # AMOS 측정 일시 데이터 확인
print(len(set(asos_all['일시']))) # 측정 일시 종류 확인

하루 단위로 측정한 데이터이며, 총 366일 동안 측정한 데이터임을 알 수 있습니다. (2024년은 윤년이었습니다.)

측정한 지점을 확인해 봅시다.

In [ ]:
# asos_all DataFrame 지점명 데이터 확인
print(sorted(list(set(asos_all['지점명'])))) # ASOS 측정 지점명 데이터 확인
print(len(set(asos_all['지점명']))) # 측정 지점명 종류 확인

울산과 인천 2개 지점에서 수집한 데이터임을 알 수 있습니다.

다음으로, 각 측정 데이터의 통계 수치를 확인해 봅시다.

In [ ]:
# asos_all DataFrame 통계수치 확인
asos_all.describe()

--- 

## 2. 데이터 정제

### 2.1. 데이터 분리

수집된 데이터를 지점별로 분리해서 별도의 변수에 저장합니다.

In [ ]:
asos_ulsan = asos_all[asos_all['지점명'] == '울산'] # 울산 지점 데이터만 추출
asos_incheon = asos_all[asos_all['지점명'] == '인천'] # 인천 지점 데이터만 추출

In [ ]:
asos_ulsan

In [ ]:
print(asos_ulsan['지점명'].unique()) # 울산 지점명 데이터 확인
print(asos_incheon['지점명'].unique()) # 인천 지점명 데이터 확인

두 데이터 모두 기준 칼럼을 바탕으로, 효과적으로 분리된 것을 알 수 있습니다.

### 2.2. 결측치 제거

데이터를 확인해 보면, `일강수량(mm)` 칼럼에 `NaN` 값이 많이 있는 것을 알 수 있습니다. 이는 결측이 누락된 결측치를 의미하며, 데이터의 특성에 맞춰 처리해 줘야 합니다.

강수량의 값이 결측된 경우는 비가 오지 않아 강수량을 측정할 수 없는 경우일 가능성이 높으므로, 이번 프로젝트에서는 강수량의 결측치를 0으로 대체합니다.

In [ ]:
# 강수량의 결측치를 0으로 대체
# loc 함수를 이용하여 강수량의 결측치가 있는 데이터의 일강수량 수치를 0으로 대체
asos_ulsan.loc[asos_ulsan['일강수량(mm)'].isnull(), '일강수량(mm)'] = 0
asos_incheon.loc[asos_incheon['일강수량(mm)'].isnull(), '일강수량(mm)'] = 0

In [ ]:
# 결측치 개수 확인
asos_ulsan.isnull().sum()

In [ ]:
asos_incheon.isnull().sum()

두 데이터 모두 결측치가 없는 것을 알 수 있습니다.

---
## 3. 종관기상관측(ASOS) 데이터 분석
공항기상관측 데이터와 교차 분석하기에 앞서, 종관기상관측 데이터를 분석해 봅시다.

### 3.1 평균기온 비교

먼저, 인천과 울산의 평균 기온을 각각 시각화해 봅시다.

In [ ]:
plt.plot(asos_incheon['일시'], asos_incheon['평균기온(°C)'])

plt.show()

In [ ]:
plt.plot(asos_ulsan['일시'], asos_ulsan['평균기온(°C)'])

plt.show()

두 데이터 모두 여름철에 기온이 높고, 겨울철에 기온이 낮은 경향을 보이지만, 두 데이터가 어떤 차이를 보이는지 비교하기 어렵습니다.

두 데이터를 한 그래프로 그려서 비교해 봅시다.

In [ ]:
# 두 데이터를 같이 그래프로 그리기
plt.plot(asos_incheon['일시'], asos_incheon['평균기온(°C)']) # 두 데이터를 모두 캔버스에 올립니다.
plt.plot(asos_ulsan['일시'], asos_ulsan['평균기온(°C)'])

plt.show() # 캔버스에 올라간 그림을 모두 보여줍니다.



두 데이터를 비교했을 때, 지역에 따라 큰 차이를 보이지 않음을 알 수 있습니다. 

그렇다면, 두 데이터의 차이값을 시각화하여 더 명확하게 분석해 봅시다.

In [ ]:
# 두 데이터의 차이를 계산하기
diff_temp = asos_incheon['평균기온(°C)'] - asos_ulsan['평균기온(°C)']

In [ ]:
diff_temp

예상대로라면 총 366개(366일)의 데이터가 나와야 하지만, 그 대신 732개의 NaN 데이터가 생겨났습니다.

이는 앞서 두 데이터를 분리하면서 Index를 초기화하지 않았기 때문에 생기는 문제입니다.

In [ ]:
print(asos_incheon.index)
print(asos_ulsan.index)

데이터를 분리하면서 원본 데이터의 index를 그대로 유지했기 때문에, 서로 겹치는 Index가 없는 두 데이터가 생성된 것을 알 수 있습니다.

두 데이터의 index를 모두 `일시` 칼럼의 값으로 초기화해 줍시다.

In [ ]:
asos_incheon.set_index('일시', inplace=True) # 일시 칼럼의 값을 index로 사용한다는 의미입니다.
asos_ulsan.set_index('일시', inplace=True)

diff_temp = asos_incheon['평균기온(°C)'] - asos_ulsan['평균기온(°C)']

이제 기온 차이 데이터가 잘 생성되었는지 확인해 봅시다.

In [ ]:
diff_temp

366개의 데이터가 문제없이 잘 생성된 것을 알 수 있습니다.

여기에 더해, `일시` 데이터를 index로 지정함에 따라 숫자 대신 측정 일시가 x축에 표시되는 것을 알 수 있습니다.

기온 차이 데이터를 생성했으니, 이를 시각화하여 두 지역의 기온 차이를 분석해 봅시다.

In [ ]:
plt.plot(diff_temp.index, diff_temp)

두 지점의 기온 차이만 시각화해 보니 앞서 두 지점의 기온 자체를 시각화 했을 때보다 기온의 차이를 더 자세히 확인할 수 있습니다.

또한 앞서 저희가 추정한 것과 달리, 절댓값 기준으로 최대 8도 정도 차이가 나는 것을 알 수 있습니다. (1월의 -8도 차이)

그렇다면, 두 지점 기온이 가장 크게 차이 난 날짜는 언제일까요?

In [ ]:
abs_diff_temp = abs(diff_temp) # 기온 차이의 절댓값을 계산합니다.

most_diff_temp = abs_diff_temp[abs_diff_temp == abs_diff_temp.max()] # 기온 차이가 가장 큰 데이터를 찾습니다.

most_diff_temp_date = most_diff_temp.index[0] # 해당 데이터의 index (일시) 값을 찾습니다.

In [ ]:
most_diff_temp_date

1월 22일이 가장 큰 기온 차이를 보였음을 알 수 있습니다.

#### 퀴즈1. 인천과 울산의 일강수량(mm) 수치의 차이가 가장 큰 날의 '일시' 값을 `most_diff_precipitation_date` 변수에 저장하세요.
- 차이는 절댓값으로 계산함에 유의하세요.

In [ ]:
precipitation_diff = asos_incheon["일강수량(mm)"] - asos_ulsan["일강수량(mm)"]
most_diff_precipitation_date = precipitation_diff.abs().idxmax()

### 3.2 기온과 해면기압의 관계 분석

다음으로, 인천과 울산 기온과 해면기압의 관계를 분석하고, 두 지점에서 비슷한 경향을 보이는지, 아니면 다른 경향을 보이는지 확인해 봅시다.

두 변수의 상관관계를 분석하는 방법은 다양하게 있지만, 저희는 여기서 피어슨 상관계수를 활용해서 상관관계를 분석해 보겠습니다.
- 피어슨 상관계수란, 두 변수 간의 선형적 관계를 계량화한 값으로, -1에서 1 사이의 값을 가집니다. 1에 가까울수록 양의 상관관계, -1에 가까울수록 음의 상관관계를 가집니다.

In [ ]:
incheon_temp_slp_corr = asos_incheon['평균기온(°C)'].corr(asos_incheon['평균 해면기압(hPa)'])
print(incheon_temp_slp_corr)

In [ ]:
ulsan_temp_slp_corr = asos_ulsan['평균기온(°C)'].corr(asos_ulsan['평균 해면기압(hPa)'])
print(ulsan_temp_slp_corr)

두 데이터 모두 -0.76 전후의 값을 나타내며, 이는 기온과 해면기압이 강한 음의 상관관계를 가진다는 것을 의미합니다.

즉, 기온이 낮은 겨울철에는 해면기압이 높은 경향을 보이며, 반대로 기온이 높은 여름철에는 해면기압이 낮은 경향을 보인다는 것을 알 수 있습니다.

정말로 그러한지 그래프를 통해 시각화해 봅시다.

In [ ]:
# 두 데이터를 같이 그래프로 그리기
plt.plot(asos_incheon['평균기온(°C)']) # 앞에서 '일시' 데이터를 index로 지정했으므로, '일시' 데이터를 x축으로 사용하지 않아도 됩니다.
plt.plot(asos_incheon['평균 해면기압(hPa)'])

plt.show() # 캔버스에 올라간 그림을 모두 보여줍니다.

두 데이터의 범위가 크게 달라서 꺾은선 그래프를 그려도 비교하기 어렵습니다. 이런 경우 정규화를 통해 범위를 일정하게 만들거나, 2차원 산점도를 그려서 비교해 볼 수 있습니다.

저희는 여기서 산점도를 그려서 비교해 보겠습니다.

In [ ]:
# 산점도 그리기
plt.scatter(asos_incheon['평균기온(°C)'], asos_incheon['평균 해면기압(hPa)'])

plt.show()

산점도로 그려보니, 기온이 높을수록 기압이 낮아지는 경향을 뚜렷하게 확인할 수 있습니다.

여기서 유의해야 할 점은, 이 분석 결과가 기온이 해면기압의 변동을 유발하거나, 반대로 해면기압이 기온의 변동을 유발하는 등 **인과관계를 의미하는 것은 아닙니다.**

#### 퀴즈2. 인천의 `일강수량(mm)` 과 `평균 해면기압(hPa)`의 상관계수를 `q2_corr` 변수에 저장하세요.

In [ ]:
q2_corr = asos_incheon["일강수량(mm)"].corr(asos_incheon["평균 해면기압(hPa)"])

선택 과제: 인천의 `일강수량(mm)` 과 `평균 해면기압(hPa)`의 관계도를 산점도로 시각화 해보세요.
- 채점에 포함된 항목은 아닙니다.

In [ ]:
plt.scatter(asos_incheon["일강수량(mm)"], asos_incheon["평균 해면기압(hPa)"])
plt.xlabel("일강수량(mm)")
plt.ylabel("평균 해면기압(hPa)")
plt.show()

---

## 4. AMOS 데이터, ASOS 데이터 교차분석

저희는 앞서 울산, 인천의 종관기상관측(ASOS) 데이터를 분석함으로써 기온과 해면기압의 추이, 그리고 그 관계성을 분석했습니다.

이제 인천공항, 울산공항의 항공기상관측(AMOS) 데이터를 같이 분석함으로써, ASOS 데이터를 분석하면서 얻은 통찰이 AMOS 데이터에도 적용되는지 확인해 봅시다.

### 4.1 AMOS 데이터 불러오기

1챕터에서 학습한 것과 같이 AMOS 데이터를 불러옵니다.

In [ ]:
# pd.read_csv를 통하여 항공기상관측(AMOS) 기상청 데이터를 데이터프레임 형태로 읽어옵니다.
amos_all = pd.read_csv("dataset/amos_rawdata.csv")
amos_all['일시'] = pd.to_datetime(amos_all['일시'])

amos_all

AMOS 데이터와 달리 ASOS 데이터는 '지점명' 칼럼으로 측정 위치를 표현합니다. 칼럼명을 하나로 통일합니다.

In [ ]:
# AMOS 데이터의 칼럼명 중 '지점명' 을 '지점' 으로 변경
amos_all.rename(columns={'지점명':'지점'}, inplace=True)

다음으로, 데이터를 두 지점으로 분할하고 index를 초기화합니다.

In [ ]:
amos_ulsan = amos_all[amos_all['지점'] == '울산공항']
amos_incheon = amos_all[amos_all['지점'] == '인천공항']

amos_ulsan.set_index('일시', inplace=True)
amos_incheon.set_index('일시', inplace=True)

### 4.2 평균기온 비교

3챕터에서 저희는 두 지점의 평균기온 추이는 비슷하며, 기온 차이는 최대 8도까지 날 수 있음을 확인했습니다.

그러나 저희가 가져온 AMOS 데이터에는 평균기온 데이터가 없으므로, 대신 최고기온과 최저기온을 바탕으로 비교해 봅시다.

In [ ]:
plt.plot(amos_ulsan['최고기온(°C)']) # 원본 데이터에 평균 
plt.plot(amos_incheon['최고기온(°C)'])

plt.show()

In [ ]:
plt.plot(amos_ulsan['최저기온(°C)']) # 원본 데이터에 평균 
plt.plot(amos_incheon['최저기온(°C)'])

plt.show()

최고기온과 최저기온 모두 AMOS 데이터와 ASOS 데이터가 비슷한 추이를 보이는 것 같습니다. 이제 같은 지역의 데이터를 비교해 봅시다.

In [ ]:
plt.plot(amos_ulsan['최고기온(°C)'])
plt.plot(asos_ulsan['최고기온(°C)'])
plt.legend(['AMOS', 'ASOS'])
plt.show()

울산의 AMOS 데이터와 ASOS 데이터의 최고기온은 거의 동일한 것을 알 수 있으나, 공항기상관측(AMOS) 데이터가 전반적으로 조금 더 높은 것으로 보입니다.

정말 수치가 다른지 확인해 봅시다.

In [ ]:
ulsan_amos_asos_temp_diff = amos_ulsan['최고기온(°C)'] - asos_ulsan['최고기온(°C)']
plt.plot(ulsan_amos_asos_temp_diff)
plt.show()

전반적으로 0~1도 정도 차이가 나는 것을 알 수 있습니다. 즉, 울산은 공항기상관측(AMOS) 이 이루어진 지점이 일반적으로 더 높은 기온을 보인다는 것을 알 수 있습니다.

인천공항도 같은 추이를 보이는지 확인해 봅시다.

In [ ]:
plt.plot(amos_incheon['최고기온(°C)'])
plt.plot(asos_incheon['최고기온(°C)'])
plt.legend(['AMOS', 'ASOS'])
plt.show()

그래프를 비교했을 때, 인천 데이터 역시 울산 데이터처럼 약간의 차이를 나타냅니다.

In [ ]:
incheon_amos_asos_temp_diff = amos_incheon['최고기온(°C)'] - asos_incheon['최고기온(°C)']
plt.plot(incheon_amos_asos_temp_diff)
plt.show()

울산 데이터와 마찬가지로 인천 데이터 역시 공항기상관측(AMOS) 데이터가 전반적으로 약간 더 높은 기온을 보인다는 것을 알 수 있습니다.

두 데이터를 비교했을 때, 공항기상관측(AMOS) 데이터와 종관기상관측(ASOS) 데이터 기온은 비슷한 추이를 보이나, AMOS 데이터의 기온이 살짝 더 높음을 확인할 수 있습니다.

이로 미루어보아, 적어도 기온 관측 면에서는 공항기상관측이 불가능할 때 종관기상관측 데이터로 어느 정도 대체할 수는 있으나, 수치에 차이가 있으므로 완전한 대체는 불가능함을 알 수 있습니다.

#### 퀴즈3. 인천의 공항기상관측(AMOS) 데이터의 최고기온이 종관기상관측(ASOS) 데이터의 최고기온보다 높거나 같은 날짜의 수를 `q3_higher_temp_count` 변수에 저장하세요.

In [ ]:
q3_higher_temp_count = int(incheon_amos_asos_temp_diff.ge(0).sum())

## 정답 확인

In [ ]:
ans = {
    "퀴즈1": str(most_diff_precipitation_date),
    "퀴즈2": float(q2_corr),
    "퀴즈3": int(q3_higher_temp_count)
}

print(ans)